<a href="https://colab.research.google.com/github/takatakamanbou/ML/blob/2025/ML2025_ex04notebookC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML ex04notebookC

<img width=72 src="https://www-tlab.math.ryukoku.ac.jp/~takataka/course/ML/ML-logo.png"> [この授業のウェブページ](https://www-tlab.math.ryukoku.ac.jp/wiki/?ML/2025)


----
## 演習: ロジスティック回帰＋勾配法によるパラメータの最適化
----




----
### 準備


以下，コードセルを上から順に実行してながら読んでいってね．

In [ ]:
# 準備あれこれ
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation, rc  # アニメーションのため
import pandas as pd
import seaborn
seaborn.set_theme()

# 「解答」を示す際に文字列を復号するのに使う
import base64
# 復号した文字列を Markdown 形式で（数式は LaTeX でフォーマットして）表示
from IPython.display import display, Markdown

In [ ]:
## 2次元正規分布で2クラスのデータを生成する関数

def getData(seed=None):

    if seed != None:
        np.random.seed(seed)

    # two 2-D spherical Gaussians
    X0 = 1.0*np.random.randn(200, 2) + [3.0, 3.0]
    X1 = 1.0*np.random.randn(200, 2) + [7.0, 6.0]
    X  = np.vstack((X0, X1))
    lab0 = np.zeros(X0.shape[0], dtype=int)
    lab1 = np.zeros(X1.shape[0], dtype=int) + 1
    label = np.hstack((lab0, lab1))

    return X, label

---
### 2クラス識別のロジスティック回帰

notebookA では，2クラスの識別問題を解くためのロジスティック回帰モデルを導入しました．次の問題で少し振り返りましょう．分からないところは notebookA で復習してね．

---
#### 問題1

次のコードセルを実行すると，notebookA の「やってみよう」と同様の実験ができます．2次元のデータを2クラスに分けるロジスティック回帰モデルを作り，パラメータを手動で適当に決めたときにモデル出力がどうなるか観察することができます．

以前の実験と同じデータを用い，そのときと同じ以下の4種類のパラメータの組に対してモデルの交差エントロピーの値がいくつになるか計算させてみましょう．

(1) 下のコードセルに以下のパラメータの組（一つ前の「やってみよう」で使ったのと同じ値）のそれぞれを入力して実行し，それぞれのときの交差エントロピーの値を求めなさい．

- (a) `(w0, w1, w2) = ( 12,  1.5, -1)`
- (b) `(w0, w1, w2) = (-12,  1.5,  1)`
- (c) `(w0, w1, w2) = ( 12, -1.5, -1)`
- (d) `(w0, w1, w2) = (-12, -1.5,  1)`

(2) 以下の文の空欄に入る語を答えなさい．$\fbox{ア}$ には(a)-(d) のいずれかが入る．$\fbox{イ}$ には「最大」「最小」「0」のいずれかが入る．

> notebookAの「やってみよう」で観察した結果，最もうまく識別できそうなのは $\fbox{ア}$ のときだった．下のコードセルを動かしてみると，この4通りのパラメータの中では，$\fbox{ア}$ のときに交差エントロピーの値が $\fbox{イ}$ になっていた．

In [ ]:
#@title パラメータの値をいろいろ変えて交差エントロピーを測ってみよう

# パラメータの例
#w0, w1, w2 = -8.2, 1.1, 0.72

w0 =  -8.2#@param {type: 'number'}
w1 =  1.1#@param {type: 'number'}
w2 =  0.72#@param {type: 'number'}

# データを生成
X, lab = getData(seed=2929)

# 等間隔にデータを作ってモデル出力を計算
xx1, xx2 = np.meshgrid(np.linspace(0, 10, num=16), np.linspace(0, 10, num=16))
XX = np.vstack((xx1.ravel(), xx2.ravel())).T
ZZ = 1.0/(1+np.exp(-(w0 + w1*XX[:, 0] + w2*XX[:, 1]))) # モデル出力を計算
zz = ZZ.reshape(xx1.shape)

# 3次元プロットの視点の設定
elevation = 20 # 上下方向の角度
azimuth = -70  # 左右方向の角度

# 3次元プロット
fig = plt.figure(facecolor='white', figsize=(8, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(X[lab==0, 0], X[lab==0, 1], 0)
ax.scatter(X[lab==1, 0], X[lab==1, 1], 1)
ax.plot_wireframe(xx1, xx2, zz, color='green')
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.view_init(elevation, azimuth)
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_zlabel('$y$')
plt.show()

# 与えられたデータに対する交差エントロピーの値を計算して出力
Z = 1.0/(1+np.exp(-(w0 + w1*X[:, 0] + w2*X[:, 1])))
H = -np.sum(np.log(Z[lab == 1])) - np.sum(np.log(1.0 - Z[lab == 0]))
print(f'パラメータ (w0, w1, w2) = ({w0:.2f}, {w1:.2f}, {w2:.2f}) のとき交差エントロピー H = {H:.2f}')

---
### 勾配法によるパラメータの最適化

----
#### 問題2

関数 $E(w_1, w_2)$ を
$$
\begin{aligned}
E(w_1, w_2) &= \frac{(2w_2-w_1^2)^2}{4} + \frac{(1-w_1)^2}{8}
\end{aligned}
$$
と定める．このとき，次のものを手計算で求めなさい．

(1) $\frac{\partial E}{\partial w_1}, \frac{\partial E}{\partial w_2}$

(2) $\frac{\partial E}{\partial w_1} = \frac{\partial E}{\partial w_2} = 0$ を満たす $(w_1, w_2)$

---

次の2つのセルを実行すると，$E(w_1, w_2)$ の概形を描かせることができます．

In [ ]:
# E(w1, w2)
def E(w):
    return (2*w[1] - w[0]*w[0])**2/4 + (1 - w[0])**2/8

# E(w1, w2) の w1, w2 に関する偏微係数
def dEdw(w):
    return np.array([0, 0]) ### 問題2で要修正

In [ ]:
# 2変数の具体例のグラフ
fig = plt.figure(facecolor='white', figsize=(12, 6))

xmin, xmax = -1, 2
ymin, ymax = -1, 2

# (w1, w2) に対する E(w1, w2) の計算
w1, w2 = np.meshgrid(np.linspace(xmin, xmax, num=100), np.linspace(ymin, ymax, num=100))
w1w2 = np.vstack((w1.ravel(), w2.ravel())).T
Ew1w2 = np.array([E(w) for w in w1w2])
EE = Ew1w2.reshape((w1.shape[0], w2.shape[1]))

# 三次元プロット
elevation = 20
azimuth = -70
ax0 = fig.add_subplot(121, projection='3d')
ax0.plot_wireframe(w1, w2, EE)
ax0.set_xlim(xmin, xmax)
ax0.set_ylim(ymin, ymax)
ax0.set_zlim(-1, 5)
ax0.set_xlabel('$w_1$')
ax0.set_ylabel('$w_2$')
ax0.set_zlabel('$E(w_1, w_2)$')
ax0.view_init(elevation, azimuth)

# 二次元等高線プロット
ax1 = fig.add_subplot(122)
cval = [0, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 1, 2, 3, 4, 5]
contour = ax1.contour(w1, w2, EE, cval)
ax1.clabel(contour, fontsize=10)
ax1.set_aspect('equal')
ax1.set_xlabel('$w_1$')
ax1.set_ylabel('$w_2$')
ax1.set_xlim(xmin, xmax)
ax1.set_ylim(xmin, xmax)

plt.tight_layout()
plt.show()

In [ ]:
# このセルを実行すると，上記の問に対する解答例が表示されます
Q = b'CigxKQokJApcYmVnaW57YWxpZ25lZH0KXGZyYWN7XHBhcnRpYWwgRX17XHBhcnRpYWwgd18xfSAmPSBcZnJhY3syfXs0fSgyd18yIC0gd18xXjIpXGZyYWN7XHBhcnRpYWwgfXtccGFydGlhbCB3XzF9KDJ3XzIgLSB3XzFeMikgKyBcZnJhY3syfXs4fSgxLXdfMSlcZnJhY3tccGFydGlhbH17XHBhcnRpYWwgd18xfSgxLXdfMSkgXFwKJj0gXGZyYWN7MX17Mn0oMndfMiAtIHdfMV4yKVx0aW1lcyAoLTJ3XzEpICsgXGZyYWN7MX17NH0oMS13XzEpXHRpbWVzICgtMSkgXFwKJj0gd18xXjMgLSAyd18xd18yICsgXGZyYWN7MX17NH13XzEgLSBcZnJhY3sxfXs0fSBcXApcZnJhY3tccGFydGlhbCBFfXtccGFydGlhbCB3XzJ9ICY9IFxmcmFjezJ9ezR9KDJ3XzIgLSB3XzFeMilcZnJhY3tccGFydGlhbCB9e1xwYXJ0aWFsIHdfMn0oMndfMiAtIHdfMV4yKSArIFxmcmFjezJ9ezh9KDEtd18xKVxmcmFje1xwYXJ0aWFsfXtccGFydGlhbCB3XzJ9KDEtd18xKSBcXAomPSBcZnJhY3sxfXsyfSgyd18yIC0gd18xXjIpXHRpbWVzIDIgKyBcZnJhY3sxfXs0fSgxLXdfMSlcdGltZXMgMCBcXAomPSAtd18xXjIgKyAyd18yClxlbmR7YWxpZ25lZH0KJCQKCigyKSAkXGZyYWN7XHBhcnRpYWwgRX17XHBhcnRpYWwgd18yfSA9IDAkIOOBqOOBiuOBj+OBqCAkd18yID0gXGZyYWN7MX17Mn13XzFeMiTvvI7jgZPjgozjgpIgJFxmcmFje1xwYXJ0aWFsIEV9e1xwYXJ0aWFsIHdfMX0gPSAwJCDjga7lvI/jgavku6PlhaXjgZfjgabmlbTnkIbjgZnjgovjgajvvIwkd18xID0gMSQg44GM5b6X44KJ44KM44KL77yOCuOBl+OBn+OBjOOBo+OBpu+8jCRcZnJhY3tccGFydGlhbCBFfXtccGFydGlhbCB3XzF9ID0gXGZyYWN7XHBhcnRpYWwgRX17XHBhcnRpYWwgd18yfSA9IDAkIOOCkua6gOOBn+OBmSAkKHdfMSwgd18yKSQKIOOBryAkKHdfMSwgd18yKSA9ICgxLCBcZnJhY3sxfXsyfSkkCg=='
display(Markdown(base64.b64decode(Q).decode('utf-8')))

----
#### 問題3

関数 $E(w_1, w_2)$ の最小値を最急降下法で求めてみよう．関数 `dEdw` を定義しているセルの `### 問題2で要修正` と書かれた行を修正してから次のセルを実行すると，$(w_1, w_2) = (0, 1)$ を初期値として $E(w_1, w_2)$ が最小になる $(w_1, w_2)$ を探すことができます．正しく動作することを確認しましょう．

ヒント: `np.array([0, 0]) ` の2箇所の `0` のうち，一つ目のところに $\frac{\partial E}{\partial w_1}$ を表す式を， 二つ目のところに $\frac{\partial E}{\partial w_2}$ を表す式を書きます．ただし，$w_1, w_2$ はそれぞれ `w[0], w[1]` です．



In [ ]:
w = np.array([0.0, 1.0]) # w_1, w_2 の初期値
eta = 0.3                # 学習係数

# 最急降下法の繰り返し
for i in range(100):
    print(f'step{i}: w = {w}, E(w) = {E(w):.4f}')
    dw = dEdw(w)  # 勾配の値の計算
    w -= eta*dw  # パラメータの更新
print(f'step{i}: w = {w}, E(w) = {E(w):.4f}')

In [ ]:
# このセルを実行すると，上記の問に対する解答例が表示されます
Q = b'CmBgYAogICAgcmV0dXJuIG5wLmFycmF5KFswLCAwXSkgIyMjIOWVj+mhjDLjgafopoHkv67mraMKYGBgCuOBruihjOOCkgpgYGAKICAgIHJldHVybiBucC5hcnJheShbd1swXSp3WzBdKndbMF0gLSAyKndbMF0qd1sxXSArIHdbMF0vNCAtIDEvNCwgLXdbMF0qd1swXSArIDIqd1sxXV0pCmBgYArjgavmm7jjgY3mj5vjgYjjgabjgYvjgonjgZ3jgozku6XpmY3jga7jgrPjg7zjg4njgrvjg6vjgpLlrp/ooYzjgZfnm7TjgZvjgbBva++8jgrmnIDmgKXpmY3kuIvms5Xjga7nubDjgorov5TjgZfoqIjnrpfjgavjgojjgaPjgaYgJCh3XzEsIHdfMikkIOOBjOW+kOOAheOBqyAkKDEsIFxmcmFjezF9ezJ9KSQg44Gr6L+R44Gl44GE44Gm44GE44GP44GT44Go44KS56K66KqN44GX44KI44GG77yOCg=='
display(Markdown(base64.b64decode(Q).decode('utf-8')))

次のセルたちを実行すると，最急降下法で $(w_1, w_2)$ の値が更新されていく様子をアニメーションで見ることができます．やってみよう．

動作確認できたら，初期値や学習係数の値をいろいろ変えて実験してみよう．

In [ ]:
def genAnim2D(f, dfdx, x0, eta):

    fig, ax = plt.subplots(facecolor="white", figsize=(6, 6))

    # f(x1, x2) の等高線を描く
    xx, yy = np.meshgrid(np.linspace(xmin, xmax, num=100), np.linspace(ymin, ymax, num=100))
    XX = np.vstack((xx.ravel(), yy.ravel())).T
    ZZ = np.array([f(x) for x in XX])
    zz = ZZ.reshape((xx.shape[0], xx.shape[1]))
    cval = [0, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 1, 2, 3, 4, 5]
    contour = ax.contour(xx, yy, zz, cval)
    ax.clabel(contour, fontsize=10)
    ax.set_aspect('equal')
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)

    # アニメーションの各コマを生成
    aList = []
    x = x0
    for i in range(50):
        a1 = ax.plot(x[0], x[1], marker='o', markersize=12, color='red')
        s = f'step{i}: $E = {f(x):.3f}$'
        a2 = ax.text(-0.9, 1.5, s, size=20)
        a1.append(a2)
        aList.append(a1)
        # 最急降下法
        dx = dfdx(x)
        x -= eta*dx

    anim = animation.ArtistAnimation(fig, aList, interval=300)
    rc('animation', html='jshtml')
    plt.close()

    return anim

In [ ]:
w = np.array([0.0, 1.0]) # w_1, w_2 の初期値
eta = 0.3                # 学習係数
anim = genAnim2D(E, dEdw, w, eta)
anim

### ゴリゴリ君の問題を勾配法で解いてみる

ゴリゴリ君のデータに直線を当てはめる線形回帰の問題の場合，連立方程式（正規方程式）を解けば最適なパラメータを一撃で求めることができます．しかし，ここではあえて勾配法を使ってみましょう．実用的な意味はなく，勾配法を理解するための実験です．

この問題では，$x$ と $y$ の値のペアが $(x_1, y_1), (x_2, y_2), \ldots, (x_N, y_N)$ と $N$ 個与えられたときに，

$$ E(a, b) = \frac{1}{2}\sum_{n=1}^N (y_n - (ax_n+b))^2 $$

の値を最小にする $(a, b)$ を求めたいのでした．第1回に導出したように，$E(a, b)$ のパラメータ $a, b$ に関する勾配は次式の通りです．

$$
\begin{aligned}
\frac{\partial E(a,b)}{\partial a} &= \sum_{n=1}^{N}(y_n-(ax_n+b))(-x_n) & (A)\\
\frac{\partial E(a,b)}{\partial b} &= \sum_{n=1}^{N}(y_n-(ax_n+b))(-1) & (B)\\
\end{aligned}
$$

したがって，$(a, b)$ を適当な値で初期化して，適当な学習係数 $\eta (>0)$ のもとで

$$
\begin{aligned}
a^\textrm{new} &= a - \eta \frac{\partial E(a,b)}{\partial a} \\
b^\textrm{new} &= b - \eta \frac{\partial E(a,b)}{\partial b} \\
\end{aligned}
$$

のように $(a, b)$ の値を更新する計算を繰り返せば，$(a, b)$ は徐々に $E(a, b)$ を最小にする解へ近づいてゆくはずです．

やってみましょう．


----
#### 問題4

式$(A), (B)$が成り立つことを示しなさい．


In [ ]:
# このセルを実行すると，上記の問に対する解答例が表示されます
Q = b'CiQkClxiZWdpbnthbGlnbmVkfQpcZnJhY3tccGFydGlhbH17XHBhcnRpYWwgYX0gRShhLCBiKSAmPSBcZnJhY3tccGFydGlhbH17XHBhcnRpYWwgYX1cbGVmdCggXGZyYWN7MX17Mn1cc3VtX3tuPTF9Xk4gICh5X24gLSAoYXhfbitiKSleMiBccmlnaHQpXFwKICY9IFxmcmFjezF9ezJ9XHN1bV97bj0xfV5OIFxmcmFje1xwYXJ0aWFsfXtccGFydGlhbCBhfSAoeV9uIC0gKGF4X24rYikpXjJcXAogJj0gXGZyYWN7MX17Mn1cc3VtX3tuPTF9Xk4gMih5X24gLSAoYXhfbitiKSlcZnJhY3tccGFydGlhbH17XHBhcnRpYWwgYX0gKHlfbiAtIChheF9uK2IpKVxcCiAmPSBcc3VtX3tuPTF9Xk4gKHlfbiAtIChheF9uK2IpKSgteF9uKQpcZW5ke2FsaWduZWR9CiQkCiRcZnJhY3tccGFydGlhbH17XHBhcnRpYWwgYn0gRShhLCBiKSQg44KC5ZCM5qeY77yOCg=='
display(Markdown(base64.b64decode(Q).decode('utf-8')))

In [ ]:
# データを読み込む
dfGori = pd.read_csv('https://www-tlab.math.ryukoku.ac.jp/~takataka/course/ML/gorigori.csv', header=0)
xmin, xmax = -5, 40
ymin, ymax = 0, 130

# データを用意
X = dfGori['気温'].to_numpy()
XX = np.vstack([X, np.ones_like(X)]).T
Y = dfGori['アイス売上数'].to_numpy()

次のセルを実行すると，上で説明した計算の過程の一例を見ることができます．

In [ ]:
a, b = 1.0, 1.0  # パラメータの初期値
nitr = 20001     # 勾配法の繰返し回数
eta = 0.001/len(X)  # 学習係数

for i in range(nitr):
    # 現在の (a, b) で予測値とその誤差を求める
    e = Y - (a*X + b)
    # 途中経過を表示
    if (i < 1000 and i % 100 == 0) or (i % 1000 == 0):
        print(f'iteration {i:>6}  a = {a:.4f}  b = {b:.4f}  E(a, b) = {e@e/2:.3f}')
    # 勾配の計算
    dEda = -e @ X
    dEdb = -np.sum(e)
    # パラメータの修正
    a -= eta * dEda
    b -= eta * dEdb


勾配法によってパラメータが最適な解へ近づいていく様子が見えるでしょうか？
ちなみに，最小二乗解は $(a, b) = (2.922, 2.337)$ （小数第3位まで表示）です．